# audiblez trên Kaggle (GPU T4 ×2): truyện audio tiếng Nga

Notebook này biến **mọi truyện** trong dataset đầu vào (`.txt`, `.md`, `.docx`, `.fb2`, `.epub`) thành `.mp4` (YouTube)
+ `.m4b`, bằng Qwen3-TTS trên **cả hai GPU T4** cùng lúc, và tự dừng gọn trước giới hạn 12 giờ của phiên.

1. Tạo một Kaggle Dataset riêng tư (vd. `audiblez-input`) chứa các file truyện, và (tuỳ chọn) thư mục giọng đọc
   `voices/<tên>/` (`reference.flac` + `reference.txt`), file từ điển phát âm `tu_dien.txt`, ảnh bìa `cover.jpg`.
2. Gắn dataset vào notebook (Add Input), chọn **Accelerator: GPU T4 x2**, bật **Internet** (cần xác minh số điện thoại).
3. Sửa ô *Cấu hình* bên dưới rồi **Save Version → Save & Run All (Commit)**: notebook chạy nền, không cần mở trình duyệt.
4. Tải kết quả ở tab *Output* (hoặc `kaggle kernels output <user>/<notebook> -p ./ket_qua`).

Chạy lại lần sau với cùng truyện: gắn *Output* của lần trước làm Input, notebook sẽ chép cache về và chỉ tổng hợp phần còn thiếu.

In [ ]:
# ==== Cấu hình ====
AUDIBLEZ = 'git+https://github.com/vansyson1308/audiblez@claude/gracious-meitner-n2so7p'
BACKEND = 'fast'                 # 'fast' = faster-qwen3-tts (CUDA graphs) | 'standard' = qwen-tts gốc
STRESS = True                    # trọng âm tự động (RUAccent + model ru-stress); False = model gốc
VOICE = 'ru_female_narrator'     # giọng có sẵn, hoặc tên thư mục trong dataset: voices/<tên>
EXTRA_ARGS = ['--youtube']       # vd. thêm '--episode-hours', '2' / '--pause-scale', '1.1'
RUN_BENCH = False                # True ở lần chạy đầu: đo cấu hình nhanh nhất trên T4 (~15-20 phút)
DTYPE = 'auto'                   # 'auto' hoặc kết quả benchmark: 'float32' / 'float16'
BATCH_SIZE = 0                   # 0 = mặc định; backend standard có thể nhanh hơn với 4 hoặc 8
SESSION_HOURS = 12.0             # giới hạn một phiên GPU của Kaggle
RESERVE_HOURS = 0.6              # để dành cho mastering, encode video và lưu output
KEEP_CHAPTER_WAVS = False        # False: xoá wav trung gian sau khi xong (nhẹ output, cache vẫn giữ)

import time
T0 = time.time()
OUT = '/kaggle/working/out'

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
import sys, torch
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Không thấy GPU: chọn Accelerator "GPU T4 x2" trong Settings của notebook.'

In [ ]:
# Cài đặt. Bản torch có sẵn của Kaggle được giữ nguyên (khớp driver CUDA của máy).
import subprocess
subprocess.run("pip freeze | grep -iE '^(torch|torchaudio|torchvision)==' > /tmp/keep_torch.txt", shell=True, check=True)
PIP = ['pip', 'install', '-q', '-c', '/tmp/keep_torch.txt']
# = the dependencies of audiblez (pyproject.toml) without qwen-tts and Kokoro
LIGHT_DEPS = ['ebooklib>=0.18,<0.19', 'soundfile>=0.13.1,<0.14', 'numpy>=1.26', 'pick>=2.4,<3', 'beautifulsoup4>=4.12',
              'lxml>=5.0', 'tabulate>=0.9,<0.10', 'librosa>=0.10', 'num2words>=0.5.13', 'pymorphy3>=2.0',
              'faster-whisper>=1.1'] + (['ruaccent>=1.6'] if STRESS else [])
if BACKEND == 'fast':
    # faster-qwen3-tts cần transformers 5 và gói qwen-tts-hf (thay cho qwen-tts): không cài qwen-tts gốc.
    # qwen-tts-hf 0.1.1.post1 chỉ nạp được model với transformers 5.15-5.16 (từ 5.17: lỗi MimiConfig.rope_theta).
    subprocess.run(PIP + ['faster-qwen3-tts', 'transformers>=5.15.1,<5.17'], check=True)
    subprocess.run(PIP + ['--no-deps', f'audiblez @ {AUDIBLEZ}'], check=True)
    subprocess.run(PIP + LIGHT_DEPS, check=True)
else:
    subprocess.run(PIP + [f'audiblez[asr{",stress" if STRESS else ""}] @ {AUDIBLEZ}'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=False, capture_output=True)
import torch
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
!audiblez --help | head -3

In [ ]:
# Tìm truyện, giọng đọc, từ điển, ảnh bìa trong dataset; chép cache của lần chạy trước (nếu có).
import glob, os, shutil
from pathlib import Path
inputs = Path('/kaggle/input')
STORY_EXT = ('.txt', '.md', '.markdown', '.docx', '.fb2', '.fb2.zip', '.epub')
stories = sorted(p for p in inputs.rglob('*') if p.is_file() and p.name.lower().endswith(STORY_EXT)
                 and '.audiblez' not in str(p) and p.name not in ('reference.txt', 'tu_dien.txt', 'lexicon.txt')
                 and 'voices' not in p.parts and not p.name.endswith(('_youtube.txt', '_parts.txt')))
voice = VOICE
for cand in inputs.rglob(f'voices/{VOICE}'):
    if cand.is_dir():
        voice = str(cand)
lexicon = next((str(p) for p in inputs.rglob('*') if p.name in ('tu_dien.txt', 'lexicon.txt')), None)
cover = next((str(p) for p in inputs.rglob('cover.*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')), None)
os.makedirs(OUT, exist_ok=True)
for old in inputs.rglob('*.audiblez'):
    if old.is_dir() and not (Path(OUT) / old.name).exists():
        shutil.copytree(old, Path(OUT) / old.name)
        print('Cache restored:', old.name)
print(f'{len(stories)} story file(s):', *[s.name for s in stories], sep='\n  ')
print('Voice:', voice, '| lexicon:', lexicon, '| cover:', cover)

In [ ]:
# (Tuỳ chọn) Benchmark: cấu hình nhanh nhất mà vẫn đạt chất lượng trên GPU này.
if RUN_BENCH:
    cmd = ['audiblez-bench', '--voice-profile', voice, '-o', '/kaggle/working/bench.json']
    if STRESS:
        cmd += ['--stress', 'ruaccent']
    subprocess.run(cmd, check=False)

In [ ]:
# Render tất cả truyện (2 GPU song song), dừng gọn trước khi hết phiên.
import json
def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    return proc.wait()

summary = []
for story in stories:
    left = SESSION_HOURS - RESERVE_HOURS - (time.time() - T0) / 3600
    if left < 0.25:
        print(f'Not enough session time left for {story.name}: run the notebook again to continue.')
        summary.append((story.name, 'not started'))
        continue
    cmd = ['audiblez', str(story), '-o', OUT, '--voice-profile', voice, '--qwen-backend', BACKEND,
           '--dtype', DTYPE, '--time-limit', f'{left:.2f}', *EXTRA_ARGS]
    if BATCH_SIZE:
        cmd += ['--batch-size', str(BATCH_SIZE)]
    if STRESS:
        cmd += ['--stress', 'ruaccent']
    if lexicon:
        cmd += ['--lexicon', lexicon]
    if cover:
        cmd += ['--cover', cover]
    t = time.time()
    code = run(cmd)
    summary.append((story.name, f'exit {code} in {(time.time() - t) / 60:.0f} min'))
print('\n'.join(f'{name}: {status}' for name, status in summary))

In [ ]:
# Dọn wav trung gian (tạo lại rất nhanh từ cache khi cần) và liệt kê kết quả.
if not KEEP_CHAPTER_WAVS:
    for d in Path(OUT).glob('*.audiblez/chapters'):
        shutil.rmtree(d, ignore_errors=True)
for p in sorted(Path(OUT).glob('*')):
    if p.is_file():
        print(f'{p.stat().st_size / 2**20:8.1f} MB  {p.name}')
for r in Path(OUT).glob('*.audiblez/qa_report.txt'):
    print('\n==', r.parent.name, '==\n' + r.read_text(encoding='utf-8')[:1500])